# Chapter 6: The Geometric Judge — Scoring Beyond Binary Correctness

**Learning objectives:**
- Understand how GMSJudge produces a GeometricVerdict with four continuous signals
- Interpret geodesic confidence, tension score and holonomy score
- See how the composite overall confidence is computed
- Run the GMS baseline and confirm 100% accuracy

---
## Preamble

Chapter 5 generated test questions with provable ground truth. This chapter shows how the **GMSJudge** scores agent answers against that ground truth — not with a binary pass/fail, but with four continuous geometric signals that capture *how right* or *how wrong* an answer is.

Key modules:
- `knowlytix.harness.testing.judge` — `GMSJudge`, `GeometricVerdict`, `create_judge`
- `knowlytix.knowledge.verify` — `check_contradictions`, `check_holonomy`
- `knowlytix.knowledge.extract` — `ExtractedTriple`, `ScoredTriple`

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from importlib.resources import files
# Load the Chapter 4 store and create the judge
import math
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig, create_judge

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=500,
    enable_tracing=False,
    device=str(DEVICE),
)
assert config.enable_tracing is False  # Plan §3 step 3 + validator C strict

bench = DOEGMSBenchmark(config)
store = bench.ingest()

judge = create_judge(store)

# Calibrate the judge up-front and propagate the calibrated thresholds onto
# judge.config so check_contradictions / check_holonomy don't raise the
# fail-loud guard. Lab 6.1 in the chapter walks through what the calibration
# does; the pre-calibration step here is a notebook-only convenience for the
# subsequent judge.judge() demonstrations.  The same wiring appears in the
# committed ch07_statistical_analysis notebook.
judge.calibrate(seed=42)
thr = judge._thresholds or {}
if thr.get("tau_contra") is not None:
    judge.config.tau_contra = float(thr["tau_contra"])
if thr.get("tau_ent") is not None:
    judge.config.tau_ent = float(thr["tau_ent"])
if thr.get("holonomy") is not None:
    judge.config.tau_path = float(thr["holonomy"])
if judge.config.tau_contra is None:
    judge.config.tau_contra = math.sqrt(2.0) + 0.2
if judge.config.tau_ent is None:
    judge.config.tau_ent = math.sqrt(2.0) / 2.0
if judge.config.tau_path is None:
    judge.config.tau_path = 2.5

print(f"\nJudge weights: {judge.weights}")
print(f"Calibrated thresholds: tau_contra={judge.config.tau_contra:.4f}, "
      f"tau_ent={judge.config.tau_ent:.4f}, "
      f"tau_path={judge.config.tau_path:.4f}")


---

## Why Binary Scoring Is Not Enough

Traditional evaluation scores an LLM answer as correct or incorrect by string matching against the ground truth. Consider this question:

> *What is the Test AUC of MDL-001?*
> Ground truth: **0.8412**

Three agent answers:
1. `"0.8412"` — **exact match** (correct)
2. `"0.84"` — **truncated** (wrong by string match, but the agent clearly knows the value)
3. `"0.95"` — **hallucinated** (wrong, and not even close)

Binary scoring treats answers 2 and 3 identically: both get a score of 0. The geometric judge distinguishes between them by measuring *how well the answer fits the manifold*.

---

## The GeometricVerdict

The `GMSJudge.judge()` method returns a `GeometricVerdict` with four continuous signals:

| Signal | Formula | Range | Interpretation |
|--------|---------|-------|----------------|
| `geodesic_confidence` | $\exp(-\bar{d})$ | $[0, 1]$ | How well the answer fits the manifold |
| `tension_score` | $\bar{E}$ | $[0, 2]$ | Mean tension energy (0 = agreement) |
| `holonomy_score` | $\bar{\delta}$ | $[0, \infty)$ | Mean holonomy defect (0 = consistent) |
| `overall_confidence` | weighted sum | $[0, 1]$ | Composite score |

### What each signal means geometrically

**Geodesic confidence** measures how well a ground truth triple fits the learned manifold. Each relation $r$ is a rotation matrix $R_r \in \mathrm{SO}(m)$ on the unit hypersphere. Given a triple $(h, r, t)$, the GMS computes the geodesic distance $d = \arccos \langle R_r \tilde{v}_h, \tilde{v}_t \rangle$ between the rotated head and the tail embedding. Confidence $= \exp(-\bar{d})$:
- $d = 0$ (0°): rotated head lands exactly on the tail → confidence = 1.0
- $d = \pi/2$ (~1.57 rad, 90°): orthogonal → confidence ≈ 0.21
- $d = \pi$ (180°): antipodal → confidence ≈ 0.04

Intuitively: *"Does the manifold think this triple belongs?"* A well-learned triple sits tightly on the surface; a rare relation may be embedded more loosely.

**Tension score** measures logical consistency between entity pairs via their polarity embeddings ($u$-space). The tension energy $E(a,b) = 2\sin(\theta/2)$ where $\theta$ is the angle between polarity vectors:
- $E \approx 0$ (θ ≈ 0°): logically aligned (**entailment**, threshold $E < 0.8$)
- $E \approx \sqrt{2} \approx 1.41$ (θ = 90°): **unrelated** ($0.8 \le E \le 1.6$)
- $E \approx 2$ (θ = 180°): logically opposed (**contradiction**, $E > 1.6$)

The "unrelated" middle range is common and healthy — it means neighboring entities serve different roles (e.g. a numeric metric and a risk rating).

**Holonomy score** captures path consistency in multi-hop reasoning. When a question traverses two or more relations ($h \xrightarrow{r_1} m \xrightarrow{r_2} t$), the judge composes the relation rotors $R_{r_2} R_{r_1}$ and compares with the direct rotor $R_r$. The defect $\delta = \| R_{\text{path}} R_r^{-1} - I \|_F$ measures how much the composed path "twists" compared to the direct route — like parallel transport around a curved surface. For single-hop questions (one triple, no path to compose), holonomy is zero by definition.

### Overall confidence

$$\text{overall} = w_{\text{exact}} \cdot \mathbf{1}_{\text{exact}} + w_{\text{geo}} \cdot \text{geodesic\_conf} + w_{\text{tens}} \cdot (1 - \bar{E}/2) + w_{\text{hol}} \cdot \exp(-\bar{\delta})$$

Each component is normalized to $[0, 1]$ before weighting. Tension $\bar{E} \in [0,2]$ is mapped to quality $(1 - \bar{E}/2)$ so low tension → high quality. Holonomy $\bar{\delta}$ is mapped via $\exp(-\bar{\delta})$ so zero defect → quality 1.0.

The five-step judge pipeline:
1. **Extract triples** from the LLM answer and question metadata
2. **Score triples** against the manifold via `score_triple()`
3. **Check contradictions** via `tension_energy()` on entity pairs
4. **Check holonomy** via rotor composition on multi-hop paths
5. **Aggregate** into the weighted overall confidence

---

## Lab 6.1 — Judge Agent Answers

Let us judge three answers to the same question: exact correct, approximate and hallucinated.

In [ ]:
# Lab 6.1: Judge three answer scenarios

meta = {"head": "MDL-001", "relation": "has_test_auc", "tail": "0.8412"}

# Scenario 1: Exact correct
v1 = judge.judge(
    llm_answer="0.8412",
    ground_truth="0.8412",
    exact_correct=True,
    exact_detail="Exact match",
    question_metadata=meta,
)

# Scenario 2: Approximate (truncated)
v2 = judge.judge(
    llm_answer="0.84",
    ground_truth="0.8412",
    exact_correct=False,
    exact_detail="Expected 0.8412, got 0.84",
    question_metadata=meta,
)

# Scenario 3: Hallucinated
v3 = judge.judge(
    llm_answer="0.95",
    ground_truth="0.8412",
    exact_correct=False,
    exact_detail="Expected 0.8412, got 0.95",
    question_metadata=meta,
)

# Summary table
print(f"{'Scenario':18s} {'Correct':>8s} {'Geodesic':>10s} {'Tension':>9s} "
      f"{'Holonomy':>10s} {'Overall':>9s}")
print("-" * 68)
for label, v in [("Exact correct", v1), ("Approximate", v2), ("Hallucinated", v3)]:
    print(f"{label:18s} {str(v.correct):>8s} {v.geodesic_confidence:>10.4f} "
          f"{v.tension_score:>9.4f} {v.holonomy_score:>10.4f} "
          f"{v.overall_confidence:>9.4f}")

### Verifying the overall confidence by hand

Let us walk through the formula for the exact-correct scenario:

$$\text{geodesic\_conf} = 0.3625, \quad \text{tension\_quality} = 1 - 1.4051/2 = 0.2974, \quad \text{holonomy\_quality} = e^{-0} = 1.0$$

$$\text{overall} = 0.3 \times \underbrace{1.0}_{\text{exact}} + 0.3 \times \underbrace{0.3625}_{\text{geodesic}} + 0.2 \times \underbrace{0.2974}_{\text{tension}} + 0.2 \times \underbrace{1.0}_{\text{holonomy}} = 0.300 + 0.109 + 0.059 + 0.200 = 0.668 \; ✓$$

For the approximate and hallucinated answers the only change is the exact-match term: $0.3 \times 0.0 = 0$ instead of $0.3 \times 1.0 = 0.3$, so their overall confidence drops by exactly 0.30 (from 0.67 to 0.37).

The geometric signals (geodesic, tension, holonomy) are identical across all three because they are computed from the *question metadata* — the same ground truth triple `(MDL-001, has_test_auc, 0.8412)` — rather than the LLM answer text. This is by design.

### Interpreting the individual signals

- **Geodesic confidence = 0.36** (distance 1.01 rad ≈ 58°). The triple `(MDL-001, has_test_auc, 0.8412)` sits at a moderate distance on the manifold. Numeric performance metrics are one of many relation types the GMS must learn, so the embedding is reasonable but not razor-tight.
- **Tension = 1.41**, squarely in the "unrelated" zone ($0.8 \le E \le 1.6$). The tail `0.8412` is checked against MDL-001's other attributes (`High`, `Degraded`, `Credit Decisioning`, etc.) which serve entirely different roles. Orthogonal polarity vectors are expected — these are *different facts about the same model*, not contradictory facts.
- **Holonomy = 0.00**. This is a single-hop question (one triple), so there is no relational path to compose. The holonomy quality contributes a full $e^{-0} = 1.0$ to the overall score.

**Key insight:** The geometric signals and the exact-match component answer two complementary questions: "How hard was this question?" (geodesic, tension, holonomy) and "Did the agent get it right?" (exact match).

### Inside the Verdict: Triples and Contradictions

Let us look at what the judge extracted and scored.

In [ ]:
# Inspect the extracted triples and contradiction checks
print("=== Verdict Anatomy (exact correct) ===\n")

print(f"Extracted triples: {len(v1.extracted_triples)}")
for et in v1.extracted_triples:
    print(f"  ({et.head}, {et.relation}, {et.tail})")

print(f"\nScored triples: {len(v1.scored_triples)}")
for st in v1.scored_triples:
    geo = f"{st.gms_score:.4f}" if st.gms_score is not None else "N/A"
    print(f"  ({st.triple.head}, {st.triple.relation}, {st.triple.tail})"
          f"  geodesic={geo}")

print(f"\nContradiction checks: {len(v1.contradictions)}")
for c in v1.contradictions:
    print(f"  ({c.entity_a:>15s}, {c.entity_b:>20s})  "
          f"tension={c.tension_energy:.4f}  contradict={c.is_contradiction}")

print(f"\nHolonomy checks: {len(v1.inconsistencies)}")
print(f"\nGeodesic confidence = exp(-{v1.scored_triples[0].gms_score:.4f})"
      f" = {v1.geodesic_confidence:.4f}")

The anatomy reveals the judge's three-stage reasoning:

**Stage 1: Triple scoring.** The judge extracted one triple `(MDL-001, has_test_auc, 0.8412)` from the question metadata. It applied the relation rotor $R_{\text{has\_test\_auc}}$ to the MDL-001 embedding and measured the geodesic distance to the `0.8412` embedding: $d = 1.0148$ rad (≈ 58°). This gives geodesic confidence $e^{-1.0148} = 0.3625$. A distance of roughly one radian is typical for well-learned numeric attributes — the GMS knows where the triple belongs on the manifold, but numeric values don't cluster as tightly as structural relations.

**Stage 2: Contradiction checks.** The judge paired the tail entity `0.8412` with five other entities connected to `MDL-001`: `CreditScore-XGB` (framework), `1.0` (threshold), `Degraded` (status), `Credit Decisioning` (use case) and `High` (risk rating). All six tension values fall in the 1.30–1.52 range. Since $\sqrt{2} \approx 1.41$ corresponds to 90° separation in polarity space, these values confirm the entities are approximately orthogonal — *unrelated*, not contradictory. This is exactly what we expect: a numeric AUC value and a risk rating like "High" describe different facets of the same model. They should not agree ($E < 0.8$) and should not contradict ($E > 1.6$). The "unrelated" zone is the healthy default.

**Stage 3: Holonomy.** No holonomy checks — this is a single-hop question. The judge only runs holonomy when the metadata specifies a multi-hop path (`r1`, `mid`, `r2` keys). With no relational triangle to close, the defect is zero and holonomy quality contributes $e^{-0} = 1.0$.

### Judging Different Question Types

The geometric signals vary by question type because different relations are learned with different fidelity on the manifold. The following experiment judges four question types — all answered correctly — to isolate how the geometric signals differ when only the relation changes.

In [ ]:
# Judging different question types
scenarios = [
    ("has_test_auc",
     {"head": "MDL-001", "relation": "has_test_auc", "tail": "0.8412"}),
    ("has_risk_rating",
     {"head": "MDL-001", "relation": "has_risk_rating", "tail": "High"}),
    ("in_section",
     {"head": "MDL-001", "relation": "in_section",
      "tail": "model_inventory_summary"}),
    ("multi-hop",
     {"head": "Accuracy", "r1": "has_model_id", "mid": "MDL-001",
      "r2": "has_approval_rate", "tail": "62.34%"}),
]

print(f"{'Question type':20s} {'Geodesic':>10s} {'Tension':>9s} "
      f"{'Holonomy':>10s} {'Overall':>9s} {'Triples':>8s}")
print("-" * 72)
for label, meta in scenarios:
    v = judge.judge(
        llm_answer="correct",
        ground_truth="correct",
        exact_correct=True,
        exact_detail="Exact match",
        question_metadata=meta,
    )
    print(f"{label:20s} {v.geodesic_confidence:>10.4f} "
          f"{v.tension_score:>9.4f} {v.holonomy_score:>10.4f} "
          f"{v.overall_confidence:>9.4f} {v.n_triples_scored:>8d}")

All four questions are answered correctly, so the exact-match term is identical ($0.3 \times 1.0 = 0.3$). The differences in overall confidence come entirely from the geometric signals:

- **`has_test_auc` (overall 0.67, geodesic 0.36):** The numeric performance metric is one of the most frequent relation types. High training frequency means the relation rotor is well-calibrated, yielding the shortest geodesic distance (~1.01 rad) and the highest geodesic confidence. Zero holonomy (single hop) lets holonomy quality contribute a full 1.0.

- **`has_risk_rating` (overall 0.64, geodesic 0.27):** A categorical attribute with fewer distinct values ("High", "Medium", "Low"). The GMS learns it reasonably well, but categorical tail embeddings are less tightly constrained than numeric ones, so the geodesic distance is slightly longer.

- **`in_section` (overall 0.47, geodesic 0.24, holonomy 1.49):** Section membership connects entities across the document hierarchy. The non-zero holonomy ($\bar{\delta} = 1.49$) appears because the judge detects a relational triangle — the same entities participate in indirect paths through other relations. The composed rotor diverges from the direct `in_section` rotor, producing a defect. Holonomy quality drops to $e^{-1.49} = 0.22$, pulling overall confidence well below the zero-holonomy relations.

- **`multi-hop` (overall 0.48, geodesic 0.25, holonomy 1.53):** This question traverses two relations: `Accuracy` → `has_model_id` → `MDL-001` → `has_approval_rate` → `62.34%`. The judge scores 3 triples (one per hop plus the head-to-tail span). The holonomy defect (1.53) measures how much the composed rotor $R_{\text{has\_approval\_rate}} R_{\text{has\_model\_id}}$ deviates from the direct head-to-tail rotor — genuine curvature in the relational geometry. Despite this, the overall confidence (0.48) remains meaningful because geodesic and tension signals are healthy.

**Note:** Tension energies cluster near $\sqrt{2} \approx 1.41$ across all four types because all questions involve `MDL-001`. The entities in its neighborhood — metrics, ratings, section labels — serve different functional roles and are approximately orthogonal in polarity space.

### Custom Weights

The default weights (0.3 exact, 0.3 geodesic, 0.2 tension, 0.2 holonomy) can be tuned. A geodesic-heavy configuration emphasizes manifold fit over binary correctness, while an exact-heavy configuration prioritizes traditional scoring.

In [ ]:
# Custom weight profiles
from knowlytix.harness.testing import GMSJudge

def _calibrated(judge):
    """Calibrate + propagate thresholds onto judge.config (mirrors cell 3)."""
    judge.calibrate(seed=42)
    thr = judge._thresholds or {}
    if thr.get("tau_contra") is not None:
        judge.config.tau_contra = float(thr["tau_contra"])
    if thr.get("tau_ent") is not None:
        judge.config.tau_ent = float(thr["tau_ent"])
    if thr.get("holonomy") is not None:
        judge.config.tau_path = float(thr["holonomy"])
    if judge.config.tau_contra is None:
        judge.config.tau_contra = math.sqrt(2.0) + 0.2
    if judge.config.tau_ent is None:
        judge.config.tau_ent = math.sqrt(2.0) / 2.0
    if judge.config.tau_path is None:
        judge.config.tau_path = 2.5
    return judge

meta = {"head": "MDL-001", "relation": "has_test_auc", "tail": "0.8412"}

weight_profiles = [
    ("Default (0.3/0.3/0.2/0.2)",
     {"geodesic": 0.3, "tension": 0.2, "holonomy": 0.2, "exact": 0.3}),
    ("Geo-heavy (0.5/0.2/0.2/0.1)",
     {"geodesic": 0.5, "tension": 0.2, "holonomy": 0.2, "exact": 0.1}),
    ("Exact-heavy (0.1/0.1/0.1/0.7)",
     {"geodesic": 0.1, "tension": 0.1, "holonomy": 0.1, "exact": 0.7}),
]

print("Approximate answer '0.84' (exact_correct=False):\n")
print(f"{'Weight profile':35s} {'Overall':>9s}")
print("-" * 46)
for label, weights in weight_profiles:
    j = _calibrated(GMSJudge(store, weights=weights))
    v = j.judge(llm_answer="0.84", ground_truth="0.8412",
                exact_correct=False, exact_detail="Expected 0.8412, got 0.84",
                question_metadata=meta)
    print(f"{label:35s} {v.overall_confidence:>9.4f}")

print("\nExact answer '0.8412' (exact_correct=True):\n")
print(f"{'Weight profile':35s} {'Overall':>9s}")
print("-" * 46)
for label, weights in weight_profiles:
    j = _calibrated(GMSJudge(store, weights=weights))
    v = j.judge(llm_answer="0.8412", ground_truth="0.8412",
                exact_correct=True, exact_detail="Exact match",
                question_metadata=meta)
    print(f"{label:35s} {v.overall_confidence:>9.4f}")


To see why the profiles diverge, consider the approximate answer `"0.84"` (exact match = 0). The geometric signals are geodesic = 0.3625, tension quality = $1 - 1.4051/2 = 0.2974$ and holonomy quality = $e^{-0} = 1.0$. Under the geo-heavy profile:

$$0.1 \times 0 + 0.5 \times 0.3625 + 0.2 \times 0.2974 + 0.2 \times 1.0 = 0 + 0.181 + 0.059 + 0.200 = 0.44$$

Under the exact-heavy profile:

$$0.7 \times 0 + 0.1 \times 0.3625 + 0.1 \times 0.2974 + 0.1 \times 1.0 = 0 + 0.036 + 0.030 + 0.100 = 0.17$$

The geo-heavy profile gives the approximate answer 0.44 while exact-heavy gives only 0.17 — a 2.6× difference. The geo-heavy profile "sees" the geometric evidence that the agent retrieved roughly the right value, whereas exact-heavy sees only the failed string match.

**Key insight:** Weight tuning is not cosmetic — it changes how the judge *defines* quality. A model risk team should select weights that match the risk tolerance of their use case.

---

## Lab 6.1b — Per-Category Calibrated Judge

The weighted-sum `overall_confidence` from Lab 6.1 mixes four signals with user-chosen weights. It is a useful inspection tool, but it does not answer the question a release gate actually asks: *is this answer correct, yes or no?* The **calibrated judge** replaces weight tuning with a principled alternative — route each question to its primary GMS primitive and use a **task-appropriate decision boundary** fitted from the store itself.

| Measure | Primitive | Decision rule (calibrated) |
|---------|-----------|-----------------------------|
| **geodesic** | `score_triple(h, r, t)` | Logistic regression $P(\text{real} \mid d)$, threshold $\tau_{\text{geo}}$ where $P = 0.5$ |
| **tension** | `tension_energy(a, b)` | 3-class logistic regression (agree / unrelated / contradict) on labels from graph structure |
| **holonomy** | `check_holonomy([r1,r2], r_direct)` | Binary logistic regression (consistent / twisted), threshold $\tau_{\text{hol}}$ fitted from real vs relation-swapped paths |
| **phase** | ENM value comparison | Direct numeric comparison of stored values |
| **enm** | ENM range check | Value $\in [v_{\min}, v_{\max}]$ from phase encoder bounds |

Two points are worth highlighting. First, the tension and holonomy thresholds are **not** the hand-picked anchors ($E<0.5$, $\delta\le 0.5$) one might guess from the geometric construction — learned embeddings drift, and an ad-hoc $\delta\le 0.5$ rule would reject essentially every real path on this store. Second, the labels are built automatically from graph structure via `knowlytix.harness.testing.primitive_calibration`: no LLM calls, no external annotation, reproducible under a seed.

`judge.calibrate()` fits all five classifiers; `judge.calibrated_classify()` then routes each question to the appropriate measure and returns a verdict with `calibration_score`, `calibration_detail` and `calibration_measure` fields.

In [ ]:
# Lab 6.1b Step 1: Calibration was completed in cell 3 (so the eager judge
# demonstrations above could exercise the contradiction + holonomy gates).
# Inspect the fitted thresholds + per-measure calibrators.
calibrations = judge._calibrations or {}
print("Calibrated measures:")
for name, clf in calibrations.items():
    print(f"  {name:14s}  classifier={type(clf).__name__}")
print("\nFitted thresholds (judge._thresholds):")
for name, tau in (judge._thresholds or {}).items():
    print(f"  {name:18s} {tau}")


### What calibrate() did

The calibration step built five classifiers from the store — no LLM calls, all labels derived from graph structure via `knowlytix.harness.testing.primitive_calibration`:

- **geodesic**: Logistic regression on `score_triple(h, r, t)` distances. Positives are real triples; negatives are tail-corrupted triples. The classifier learns where the rotated-head-to-tail distance separates real from spurious triples.
- **path_transport**: Same signal for two-hop paths via `transport.path_score(h, [r1, r2], t)`. Composed rotors accumulate curvature error so the distribution is systematically shifted; a separate classifier captures the wider boundary.
- **tension**: Three-class logistic regression on `tension_energy(a, b)`. Labels come from graph structure — **agree** = pairs sharing a `(relation, target)`, **unrelated** = random pairs with no shared target or direct edge, **contradict** = pairs linked through opposing relations (e.g. `complies_with` vs `violates`). When the schema has no explicit opposition vocabulary, the labeler returns two classes and the judge overlays a geometric anchor at $E=1.6$ so extreme values still flag contradict.
- **holonomy**: Binary logistic regression on rotor composition defect $\|R_{r_2} R_{r_1} R_{r_{\text{direct}}}^{-1} - I\|_F$. Positives are **consistent** paths — real $(h, r_1, m), (m, r_2, t)$ triples where the direct edge $(h, r_{\text{direct}}, t)$ also exists. Negatives are **twisted** paths, obtained by replacing $r_2$ with a random relation on the same $(h, m, t)$ nodes.
- **phase**: Logistic regression on phase encoder angles for ENM values. Positives are stored values; negatives are perturbed values outside the encoder's range.

Each fit reports the sample count, class breakdown, in-sample accuracy, and — for binary fits — the interpretable threshold $\tau$ where $P(\text{accept}) = 0.5$. The thresholds are stored on `judge._thresholds` and used by `_tension_label_from_value()` and `_holonomy_threshold()` at inference time.

In [ ]:
# Lab 6.1b Step 2: Evaluate positive + negative on 130 base questions
import random
from knowlytix.benchmark.eval.scorers import score_answer
from knowlytix.harness.testing.bridge import generate_base_questions

random.seed(42)
entities = list(store.adapter.entity_to_idx.keys())

base_qs = generate_base_questions(store, max_per_category=10, seed=42)

# Per-category counters
cat_pos_ok = Counter()   # positive correctly accepted
cat_pos_n = Counter()    # total positives
cat_neg_ok = Counter()   # negative correctly rejected
cat_neg_n = Counter()    # total negatives
cat_measure = {}         # measure used per category

for q in base_qs:
    meta = q.metadata
    gt = str(q.ground_truth)
    cat = q.category
    ans_type = q.answer_type

    # --- Positive: ground-truth answer ---
    s_gt = score_answer(gt, q.ground_truth)
    v_pos = judge.judge(
        llm_answer=gt, ground_truth=gt,
        exact_correct=s_gt.correct, exact_detail=s_gt.detail,
        question_metadata=meta)
    cal_pos = judge.calibrated_classify(v_pos, cat, gt, meta)

    cat_pos_n[cat] += 1
    if cal_pos.calibrated_verdict:
        cat_pos_ok[cat] += 1

    # Record which measure was used
    if cat not in cat_measure:
        cat_measure[cat] = cal_pos.calibration_measure

    # --- Negative: corrupted answer ---
    low_gt = gt.lower().strip()
    if ans_type in ("bool",) or low_gt in ("true", "false"):
        fake = "False" if low_gt == "true" else "True"
    elif low_gt in ("a", "b"):
        fake = "B" if low_gt.upper() == "A" else "A"
    elif low_gt in ("agree", "contradict", "unrelated"):
        labels = ["agree", "contradict", "unrelated"]
        fake = random.choice([l for l in labels if l != low_gt])
    else:
        fake = random.choice(entities)
        while fake == gt:
            fake = random.choice(entities)

    s_fake = score_answer(fake, q.ground_truth)
    v_neg = judge.judge(
        llm_answer=fake, ground_truth=gt,
        exact_correct=s_fake.correct, exact_detail=s_fake.detail,
        question_metadata=meta)
    cal_neg = judge.calibrated_classify(v_neg, cat, fake, meta)

    cat_neg_n[cat] += 1
    if not cal_neg.calibrated_verdict:  # correctly rejected
        cat_neg_ok[cat] += 1

# Print per-category table
cats = sorted(cat_pos_n.keys())
total_pos_ok = sum(cat_pos_ok[c] for c in cats)
total_pos_n = sum(cat_pos_n[c] for c in cats)
total_neg_ok = sum(cat_neg_ok[c] for c in cats)
total_neg_n = sum(cat_neg_n[c] for c in cats)

print(f"{'Category':25s} {'Pos':>6s} {'Neg':>8s} {'Acc':>7s}   {'Measure'}")
print("-" * 65)
for cat in cats:
    p_ok = cat_pos_ok[cat]
    p_n = cat_pos_n[cat]
    n_ok = cat_neg_ok[cat]
    n_n = cat_neg_n[cat]
    acc = (p_ok + n_ok) / (p_n + n_n) * 100
    measure = cat_measure.get(cat, "?")
    print(f"{cat:25s} {p_ok:>2d}/{p_n:<3d} {n_ok:>3d}/{n_n:<4d} {acc:>5.1f}%   {measure}")

total_acc = (total_pos_ok + total_neg_ok) / (total_pos_n + total_neg_n) * 100
print("-" * 65)
print(f"{'TOTAL':25s} {total_pos_ok:>2d}/{total_pos_n:<3d} "
      f"{total_neg_ok:>3d}/{total_neg_n:<4d} {total_acc:>5.1f}%")

# Count categories at 100%
perfect = sum(1 for c in cats
              if (cat_pos_ok[c] + cat_neg_ok[c]) == (cat_pos_n[c] + cat_neg_n[c]))
print(f"\nCategories at 100%: {perfect}/{len(cats)}")

### Analysis

**98.5% paired accuracy** (129/130 positive accepted, 127/130 negative rejected) across 13 categories, using five distinct geometric measures with no fallbacks and no exact-match dependency. Per-question correctness (1 iff both halves of the pair route the right way) is **97.7% with BCa 95% CI [93.8%, 99.2%]** from `lab_artifacts/ch06_calibration.json -> verdict_accuracy.overall`.

**10 of 13 categories at 100%.** Two categories land at 95% (one borderline case each: `adversarial`, `procedural`). These reflect the 99.9% single-hop geodesic calibration budget — a handful of corrupted entities happen to sit near the logistic decision boundary at $m=128$.

**The holonomy category lands at 90%** (9/10 positive accepted, 9/10 negative rejected). The `HolonomyGenerator` ships **structural-truth labels**: a *real triangle* (all three edges in the graph) is labeled `True`; a *corrupted triangle* (the direct edge has been swapped to a relation that does not close the triangle) is labeled `False`. The judge's calibrated threshold $\tau_{\text{hol}}=10.62$ (BCa 95% CI $[10.41, 10.78]$) discriminates the two structural classes at $\sim$85% on the calibration cohort; the 90% on the 10/10 sub-cohort sits inside that confidence interval. The labels are anchored on graph structure, not on a defect cutoff — threshold discrimination is the judge's role, not the generator's.

**Per-category routing is the operating contract.** The calibrated judge uses only the GMS manifold and routes each question to its measure-of-record — there is no four-signal weighted aggregation:
- **geodesic** handles most categories (plausibility, conditional, procedural, robustness, recovery, governance, adversarial, multi-hop transport) via a single logistic regression on triple distance
- **tension** uses tension energy directly with the calibrated $\tau_{\text{ent}}$ for tension-category questions; the runtime overlay `_TENSION_CONTRADICT_ANCHOR` flags extreme contradict values when the corpus has no `(r_a, opposite_of, r_b)` triples (the canonical `signal_unavailable` state)
- **holonomy** uses binary logistic regression on rotor composition defect for holonomy-category questions
- **phase** uses direct ENM value comparison for phase comparison questions
- **enm** uses range checking for boundary questions (and ENM cross-checks for numeric link-prediction tails)

In [ ]:
# Lab 6.1b Step 3: Decision detail report — one example per measure
print("=== Decision Detail Report (one per measure) ===\n")

shown_measures = set()
for q in base_qs:
    meta = q.metadata
    gt = str(q.ground_truth)
    cat = q.category

    s_gt = score_answer(gt, q.ground_truth)
    v = judge.judge(
        llm_answer=gt, ground_truth=gt,
        exact_correct=s_gt.correct, exact_detail=s_gt.detail,
        question_metadata=meta)
    cal = judge.calibrated_classify(v, cat, gt, meta)
    measure = cal.calibration_measure

    if measure and measure not in shown_measures:
        shown_measures.add(measure)
        verdict_str = "ACCEPT" if cal.calibrated_verdict else "REJECT"
        print(f"  [{measure:12s}] {cat:25s}")
        print(f"    Answer:  {gt[:60]}")
        print(f"    Score:   {cal.calibration_score}")
        print(f"    Detail:  {cal.calibration_detail}")
        print(f"    Verdict: {verdict_str}")
        print()

    if len(shown_measures) >= 5:
        break

### Reading the detail lines

Each `calibration_detail` string shows the geometric primitive that made the decision:

- **geodesic**: `geodesic=1.06, p_real=0.99 -> ACCEPT` means the rotated head lands within 1.06 radians of the tail embedding on the hypersphere. The logistic regression gives 99% confidence this is a real triple — well above the 0.5 threshold, so the answer is accepted.

- **tension**: `E(A,B)=1.29 -> unrelated` means the polarity angle between entities A and B is approximately 76 degrees, placing the tension energy midway between agreement (0 degrees, $E < 0.5$) and contradiction (180 degrees, $E > 1.5$). The LLM's classification label must match this geometric determination.

- **holonomy**: `delta=2.61 -> inconsistent` means the composed rotor $R_{r_2} R_{r_1}$ deviates significantly from the direct rotor $R_{\text{direct}}$, with a Frobenius norm defect of 2.61 (well above the 0.5 consistency threshold). Like parallel transport around a curved surface, the relational path "twists" too much to be consistent.

- **enm**: `ENM: 0.5000 in [0.0, 1.0] -> in range` means the test value (0.5) falls within the phase encoder's bounds $[v_{\min}, v_{\max}]$. The ENM stores exact numeric values on $S^1$ — this is a direct lookup, not an approximation.

- **phase**: `ENM: A=67, B=87 -> B larger` means the ENM values for entities A and B are compared directly. No regression, no threshold — just which stored value is larger.

In [ ]:
# Lab 6.1b Step 4: Category-to-measure mapping table
print(f"{'Category':25s} {'Measure':>12s} {'Diagnostic':>14s}")
print("-" * 55)
for cat in sorted(judge.CATEGORY_MEASURE.keys()):
    m = judge.CATEGORY_MEASURE[cat]
    d = judge.CATEGORY_DIAGNOSTIC[cat]
    print(f"  {cat:25s} {m:>10s} {d:>14s}")

The mapping table shows how each of the 13 question categories is routed to a geometric measure. The **Measure** column determines the classifier or threshold logic used for the accept/reject decision. The **Diagnostic** column shows the geometric primitive the category was designed to test — this is the conceptual signal, even when the classification itself uses a different measure.

For example, the `tension` category tests tension energy as a *diagnostic* (the question asks about agreement/contradiction), but the `calibrated_classify()` method uses `tension_energy()` directly with threshold logic rather than the geodesic logistic regression. Similarly, `holonomy` questions test path consistency and use `check_holonomy()` directly.

Most categories route to `geodesic` because they ultimately ask "is this triple real?" — and the geodesic distance on the manifold is the principled measure for that question. The exceptions (tension, holonomy, phase, enm) each have their own geometric primitive that provides a more direct answer than geodesic distance alone.

---

### GMSJudge vs. LLM-as-Judge

How does the geometric judge compare with the increasingly popular pattern of using a second LLM to grade agent answers? We evaluate both approaches on the same 130 base questions (ground-truth positive + corrupted negative):

| Judge | Accuracy | FAR | FRR | Latency |
|-------|----------|-----|-----|---------|
| **Calibrated GMSJudge** | **98.1%** | **2.3%** | **1.5%** | **<1 s** |
| LLM-as-judge (grounded) | 53.1% | 80.0% | 14.6% | ~9 min |

The LLM judge accepts 80% of wrong answers (FAR = 80%) — four out of five corrupted answers pass. The core problem is *anchoring*: the LLM sees the grounding document, finds plausible-sounding context for the corrupted entity, and declares it correct. Without a geometric ground truth, the LLM has no way to distinguish a real triple from a corrupted one that "sounds right."

The GMSJudge eliminates this failure mode entirely:
- **Deterministic.** Same input always produces the same verdict — no temperature, no prompt sensitivity.
- **Auditable.** Each verdict carries the raw geometric score ($d$, $E$, $\delta$, ENM value) and a one-line explanation. A reviewer can verify the decision without re-running the judge.
- **Fast.** Sub-second on CPU; no API calls. The LLM judge requires 260 API calls (130 answering + 130 judging) and ~9 minutes of wall time.
- **No ground-truth leakage.** The GMSJudge classifies using geometric primitives calibrated from the store's own training triples — it never sees the exact-match label.

---

## Lab 6.2 — Run GMS Baseline

The **GMS baseline** answers every question using the GMS answer function directly — the same function used to generate the ground truth. It should score 100% accuracy. If it does not, there is a bug in the ground truth pipeline.

In [ ]:
# Lab 6.2: Run GMS baseline
result = bench.run_gms_baseline()

print(f"\n{'='*55}")
print("  GMS Baseline Results")
print(f"{'='*55}")
print(f"  Total questions:  {result.total_questions}")
print(f"  Total correct:    {result.total_correct}")
print(f"  Accuracy:         {result.total_correct / result.total_questions * 100:.1f}%")
print(f"  Model:            {result.model}")

In [ ]:
# Category-level accuracy
cat_total = Counter()
cat_correct = Counter()
for qr in result.questions:
    cat_total[qr.category] += 1
    if qr.correct:
        cat_correct[qr.category] += 1

print(f"{'Category':25s} {'Correct':>8s} {'Total':>6s} {'Accuracy':>9s}")
print("-" * 52)
for cat in sorted(cat_total.keys()):
    acc = cat_correct[cat] / cat_total[cat] * 100
    print(f"{cat:25s} {cat_correct[cat]:>8d} {cat_total[cat]:>6d} {acc:>8.1f}%")

# Check for failures
failures = [qr for qr in result.questions if not qr.correct]
if failures:
    print(f"\nFailures: {len(failures)}")
    for f in failures[:5]:
        print(f"  {f.doe_qid}: expected={f.ground_truth[:40]}, "
              f"got={f.answer[:40]}")
else:
    print(f"\nAll {result.total_questions} questions answered correctly.")

---

## Exercises

### Exercise 6.1

Re-run `judge.calibrate()` with a different seed (`seed=7`) and compare the fitted thresholds to the `seed=42` baseline. Which measure is most stable under resampling? Which is least stable, and why?

### Exercise 6.2

Run the calibrated judge on 10 plausibility questions with both the correct answer and a deliberately wrong answer. Collect `calibration_score` (geodesic distance) for each case into two lists. Does the geodesic distribution cleanly separate real from corrupted answers, or are there borderline cases near $\tau_{\text{geo}}$?

### Exercise 6.3

Examine the tension energy values from the contradiction checks in the verdict anatomy. Which entity pair has the highest tension? Look up both entities in the store and explain why they are geometrically distant.

### Exercise 6.4 — Inspecting your own calibration

Calibration is **use-case specific**: the fitted thresholds depend entirely on the schema and density of your own store. Use the `primitive_calibration` library to inspect the per-class distributions for your calibration run and sanity-check them against the geometric anchors ($E=\sqrt{2}\approx 1.414$ for orthogonal polarities, $\delta=0$ for exact commutation). Use the starter below.


In [ ]:
# === Exercise 6.1: Your code here ===
# Re-run judge.calibrate() with a different seed and inspect how the
# fitted thresholds change. A stable calibration should produce
# thresholds that vary only within the confidence interval implied by
# the sample size. Compare τ_geo, τ_tension, τ_holonomy for seed=42
# vs seed=7.
#
# Starter:
# j7 = GMSJudge(store); _ = j7.calibrate(seed=7)
# for name in ['geodesic', 'tension', 'holonomy']:
#     tau42 = judge._thresholds.get(name)
#     tau7  = j7._thresholds.get(name)
#     print(f'{name:10s}  seed=42 τ={tau42}  seed=7 τ={tau7}')

In [ ]:
# === Exercise 6.2: Your code here ===


In [ ]:
# === Exercise 6.3: Your code here ===


In [ ]:
# === Exercise 6.4: Inspect your own calibration distributions ===
from knowlytix.harness.testing.primitive_calibration import (
    label_tension, label_holonomy, summarize_samples
)

tension_samples = label_tension(store, seed=42)
holonomy_samples = label_holonomy(store, seed=42)

print('Tension samples (per class):')
for name, s in summarize_samples(tension_samples).items():
    print(f"  {name:10s} n={s['n']:4d}  mean={s['mean']:.3f}  "
          f"std={s['std']:.3f}  range=[{s['min']:.3f}, {s['max']:.3f}]")
print()
print('Holonomy samples (per class):')
for name, s in summarize_samples(holonomy_samples).items():
    print(f"  {name:10s} n={s['n']:4d}  mean={s['mean']:.3f}  "
          f"std={s['std']:.3f}  range=[{s['min']:.3f}, {s['max']:.3f}]")
print()
print('Fitted thresholds from judge.calibrate() above:')
for name, tau in judge._thresholds.items():
    print(f'  {name:14s} tau={tau}')

# Sanity check questions to consider:
#   * Does the unrelated tension cluster sit near sqrt(2) ≈ 1.414?
#   * Is the fitted tension threshold below or above your agree mean?
#   * Does the consistent holonomy mean fall well below the twisted mean?


---

## Summary

In this chapter we explored the geometric judge — the component that turns raw agent answers into multi-dimensional verdicts:

1. **Binary scoring is insufficient.** String matching treats "0.84" and "0.95" identically when the ground truth is "0.8412." The geometric judge provides continuous confidence scores that distinguish near-misses from hallucinations.
2. **Four geometric signals** capture different aspects of answer quality: geodesic confidence (manifold fit), tension score (consistency), holonomy score (path coherence) and the exact-match component (traditional correctness).
3. **The five-step pipeline** extracts triples from the answer, scores them against the manifold, checks contradictions via tension energy, verifies path consistency via holonomy and surfaces a per-signal verdict for downstream routing.
4. **Per-category routing is the operating contract.** Each question routes to its primary GMS primitive with a fitted threshold; the legacy weighted-sum `overall_confidence` field is an inspection tool, not a release-gate primitive. The judge does not pool the four geometric signals into a single overall score — pooling is unit-incoherent across the four signals (geodesic in radians, tension in $[0, 2]$, holonomy in unbounded rotor-composition defect, exact match binary), and the canonical pattern from `gms_agent_builder_ext` matches each question to one primitive of record.
5. **Task-appropriate calibration.** The three calibrated decision measures fit logistic regressions against labels derived from graph structure via `knowlytix.harness.testing.primitive_calibration`: real vs. corrupted triples for geodesic ($\tau=0.905$, BCa 95% CI [0.865, 0.930], 99.9%), agree/unrelated pairs for tension ($\tau_{\text{ent}}=0.658$, BCa 95% CI [0.645, 0.672]), and `HolonomyGenerator`-emitted real vs. corrupted triangles for holonomy ($\tau_{\text{hol}}=10.62$, BCa 95% CI [10.41, 10.78], 84.7%). The contradict cut $\tau_{\text{contra}}$ is recorded as `signal_unavailable` — the canonical `bootstrap_policy_tau_contra` returns `None` when the corpus has no `(r_a, opposite_of, r_b)` triples; the runtime overlay `_TENSION_CONTRADICT_ANCHOR` flags extreme tension values, and the policy layer handles the missing signal via quorum.
6. **98.5% paired accuracy on 130 positive/negative probes.** The routed judge correctly classifies 129/130 positives and 127/130 negatives, with 10 of 13 categories at 100%. Per-question correctness sits at 97.7% with BCa 95% CI [93.8%, 99.2%] from `lab_artifacts/ch06_calibration.json -> verdict_accuracy.overall`. The holonomy category lands at 90% because the calibrated manifold threshold $\tau_{\text{hol}}=10.62$ separates the real-triangle and corrupted-triangle distributions at $\sim$85% on the calibration cohort; the 10/10-per-side lab sample sits inside the BCa CI. Generator labels are structural (graph-derived), so the lab measures the judge's discrimination accuracy directly.
7. **GMSJudge vs. LLM-as-judge.** The calibrated GMSJudge (98.5% paired accuracy, <1 s) substantially outperforms LLM-as-judge (53.1% accuracy, 80% FAR, ~9 min). The LLM judge anchors on plausible-sounding context and accepts four out of five wrong answers.
8. **The GMS baseline** confirms the internal consistency of the testing infrastructure — 520/520 questions answered correctly across all 13 categories.

**Next:** Chapter 7 shows how to analyze factor effects via logistic regression — turning the 520 DOE verdicts into actionable insights about which presentation factors affect agent accuracy.